AI-powered brochure generator that scrapes and navigates company websites intelligently.

In [15]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
from IPython.display import Markdown, display, update_display
from scrapper import fetch_website_contents,fetch_website_links
load_dotenv(override=True)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(base_url=GEMINI_BASE_URL, api_key=os.getenv("GEMINI_API_KEY"))


Step 1 Link Extractions and filter useful links for brouchers

In [18]:
links=fetch_website_links("https://edwarddonner.com")
print(links)

['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/

In [19]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [20]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite", 
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [22]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'professional profile',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [33]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [26]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [32]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-H3
Updated
about 22 hours ago
•
59.4k
•
3.48k
meta-models/Muse-Glimmer-30B
Updated
about 9 hours ago
•
879
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
10 days ago
•
1.05M
•
3.1k
Comfy-Org/MiniMax-H3
Updated
2 days ago
•
6.8M
•
1.17k
larryvrh/MiniMax-H3-Turbo-Lora
Updated
3 days ago
•
613
Browse 2M+ models
Spaces
Running
on
Zero
MCP
829
Wan2.2 14B Fast Preview [NEW

Step 2: Broucher Creation

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-H3\nUpdated\nabout 22 hours ago\n•\n59.4k\n•\n3.48k\nmeta-models/Muse-Glimmer-30B\nUpdated\nabout 

In [34]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [35]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

## Company Overview
**Hugging Face** is the premier collaboration platform for the machine learning community. It is the central hub where developers, researchers, and enterprises come together to create, discover, and collaborate on models, datasets, and applications. 

By empowering open-source innovation, Hugging Face provides the tools and infrastructure needed to move faster and build the future of artificial intelligence.

---

## What We Offer

### Platform Features
* **Models (2M+):** Discover, host, and fine-tune state-of-the-art machine learning models across vision, audio, text, and multimodal AI.
* **Datasets (500k+):** Access and share massive, high-quality open datasets for training and benchmarking.
* **Spaces (1M+ Applications):** Build, host, and showcase interactive AI demos and web applications seamlessly.
* **Storage Buckets & Inference:** Dedicated Storage Buckets and high-performance Inference Endpoints for scalable deployment.

### Enterprise & Business Solutions
* **Team & Enterprise Plans:** Dedicated support, enhanced security, and private collaboration environments for organizations.
* **Hugging Face PRO:** Upgraded tools and compute options for individual creators and power users.
* **Enterprise Support & Infrastructure:** Optimized inference providers, specialized support, and tailored compute solutions to power enterprise-grade AI workloads.

---

## Our Community & Impact
Hugging Face serves as the home of the global open-source AI movement. The platform connects millions of users through:
* **Knowledge Sharing:** A vibrant platform featuring Daily Papers, technical blog posts, research articles, and educational tracks such as Hugging Face Fundamentals.
* **Open Collaboration:** Direct integration with GitHub, active community forums, and a thriving Discord community.
* **Cutting-Edge Research:** Leadership in open-source AI developments, including AI agent orchestration, agentic reinforcement learning, and modern model architectures.

---

## Life & Culture at Hugging Face

### Culture & Values
* **Open Source First:** We believe in the power of open collaboration, transparency, and democratizing access to state-of-the-art machine learning.
* **Innovation & Agility:** Our team moves quickly, pioneering breakthroughs in AI agents, inference optimization, and developer tooling.
* **Continuous Learning:** We prioritize knowledge-sharing through community articles, daily paper discussions, and educational initiatives.

### Careers & Team
With a growing global team of over 180+ experts, Hugging Face attracts top talent in machine learning engineering, software development, research, and open-source community building. 

Whether you are looking to deploy enterprise AI solutions, invest in the next generation of open technology, or build your career at the center of the AI revolution, Hugging Face offers an environment where impactful work shapes the future of technology.

---

## Get Started
* **Explore the Hub:** Visit huggingface.co to browse 2M+ models and 1M+ applications.
* **Solutions for Organizations:** Learn more about Enterprise plans and Inference Endpoints to supercharge your company's AI capabilities.

Step 3: Adding streaming support to the brochure generation process

In [40]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [41]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Hugging Face is the leading collaboration platform for the machine learning community. Known as the home of modern open-source AI, Hugging Face enables developers, researchers, and organizations to create, discover, and collaborate on cutting-edge ML models, datasets, and applications.

---

## The Machine Learning Hub

Hugging Face hosts the world’s largest open ecosystem for artificial intelligence:

* **2M+ Models:** Explore, test, and deploy state-of-the-art models for text, vision, audio, and multimodal tasks.
* **500k+ Datasets:** Access diverse, community-curated datasets driving modern AI research and training.
* **1M+ Applications (Spaces):** Discover interactive AI demos, custom applications, and agentic workflows built by creators worldwide.
* **Core ML Tools:** Includes HuggingChat, Inference Endpoints, Storage Buckets, and Daily Papers to accelerate development workflows.

---

## Enterprise & Business Solutions

Hugging Face provides enterprise-grade infrastructure and security so organizations can move faster and deploy AI with confidence.

### Offerings for Businesses & Enterprise Customers:
* **Team & Enterprise Plans:** Host and collaborate securely on proprietary ML assets with advanced organization management.
* **Inference Endpoints & Providers:** Turn models into production-ready APIs with dedicated compute and tailored performance.
* **Storage Buckets:** Built-in high-performance cloud storage customized for ML asset management and agent workflows.
* **Enterprise Support & Hugging Face PRO:** Dedicated expertise, priority resources, and developer tools to power enterprise AI roadmaps.

---

## Culture & Community

At its core, Hugging Face is defined by a mission to democratize open-source AI and build transparent, accessible technology. 

* **Open-Source Pioneer:** Hugging Face acts as a central catalyst for global open-source AI research—fostering rapid innovation in agent orchestration, reinforcement learning, and open model architectures.
* **Active Ecosystem:** Supported by a thriving global developer community connected across Discord, interactive forums, GitHub, and learning tracks such as *Hugging Face Fundamentals*.
* **Thought Leadership:** Regular publications of daily papers, ecosystem updates, and technical articles keep the community at the forefront of AI innovation.

---

## Careers & Team Growth

Hugging Face is powered by a world-class global team (185+ team members and growing) collaborating directly with millions of open-source contributors.

### Why Join or Partner with Hugging Face?
* **High Impact:** Shape the core infrastructure driving the future of open-source and enterprise AI.
* **Collaborative Environment:** Work alongside leading engineers, researchers, and creators across the global ML community.
* **Innovation-First:** Build next-generation AI tools, agent orchestration systems, and platform services that redefine how software is built.

---

*Ready to collaborate, deploy, or join the team? Explore more at [huggingface.co](https://huggingface.co).*